In [1]:
import torch

In [2]:
B, T, d = 100, 200, 500
x = torch.rand(B,T,d)
y = torch.zeros(d)

In [3]:
y = y.unsqueeze(0).unsqueeze(0).expand(B,1,d)
z = torch.cat((y,x), dim=1)
z.shape # 100, 201, 500

torch.Size([100, 201, 500])

In [4]:
A = torch.rand(100,100)
B = torch.rand(100,100)
C = torch.stack((A,B), dim=0)
C.shape # 2 x 100 x 100 ; new dim!

torch.Size([2, 100, 100])

In [5]:
# testing key bias and key/value bias 
from model import GPT, GPTConfig
import torch
B,T= 1, 3
for type in ["standard", "key_bias", "key_value_bias"]:
    config_normal = GPTConfig(attention_type=type, n_layer=1)
    model = GPT(config_normal)
    dummy_batch = torch.ones(size=(B,T), dtype=torch.long)
    output, _ = model(dummy_batch)

number of parameters: 45.72M
torch.Size([1, 12, 3, 64]) | torch.Size([1, 12, 3, 64]) | torch.Size([1, 12, 3, 64])
mask: torch.Size([3, 3])
error rate: 1.5072990208864212e-05
number of parameters: 45.72M
torch.Size([1, 12, 3, 64]) | torch.Size([1, 12, 4, 64]) | torch.Size([1, 12, 4, 64])
mask: torch.Size([3, 4])
error rate: 1.951365265995264e-05
number of parameters: 45.72M
torch.Size([1, 12, 3, 64]) | torch.Size([1, 12, 4, 64]) | torch.Size([1, 12, 4, 64])
mask: torch.Size([3, 4])
error rate: 7.812923286110163e-06


In [31]:
# comparing 
B = 1
T = 3
d = 25
Q=K=V = torch.rand(B,T,d)

mask = torch.tril(torch.ones(size=(T,T)))
att = ((Q @ K.transpose(1,2)) / 5).masked_fill(mask == 0, float("-inf"))
att = torch.softmax(att, dim=-1)
out = att @ V

In [32]:
y = torch.nn.functional.scaled_dot_product_attention(Q,K,V, is_causal=True)
y.shape

torch.Size([1, 3, 25])

In [33]:
print(f"error rate: {torch.max(torch.abs(out - y))}")

error rate: 1.1920928955078125e-07


### Testing model returning attention distribution during inference and not during training